## views מול copies

זהו אחד ממקורות הבאגים הנפוצים ביותר בעבודה עם NumPy. כאשר פורסים מערך (‎`a[2:5]`‎), NumPy **לא** יוצר מערך חדש בזיכרון — הוא מחזיר **תצוגה (view)** שמצביעה על אותו זיכרון. שינוי בתצוגה משנה גם את המערך המקורי, ולהיפך.

זה שונה מרשימות פייתון, שבהן פריסה (‎`lst[2:5]`‎) תמיד יוצרת רשימה חדשה (עותק). כדי לקבל התנהגות דומה במערך NumPy יש לבקש עותק במפורש: ‎`a[2:5].copy()`‎.

### דוגמה: "ניקוי" נתוני מדידה

יש לנו יומן מדידות גובה גולמי מניסוי נפילה חופשית, וברצוננו לתקן רק את שלוש המדידות האמצעיות (שהיו שגויות עקב רעש רגעי במכשיר):

In [ ]:
import numpy as np

raw_log = np.array([20.0, 18.8, -999.0, -999.0, -999.0, 5.2, 0.1])  # -999 = קריאה פגומה
print("לפני:", raw_log)

middle = raw_log[2:5]     # זו תצוגה, לא עותק!
middle[:] = [17.6, 12.4, 8.9]   # "מתקנים" את התצוגה

print("אחרי:", raw_log)   # היומן הגולמי עצמו השתנה, דרך התצוגה

בשורת "לפני" NumPy הדפיס את המערך בכתיב מדעי, משום שיש בו מספרים בסדרי גודל שונים מאוד: ‎`2.00e+01`‎ פירושו 20, ו-‎`1.00e-01`‎ פירושו 0.1.

במקרה הזה זה בדיוק מה שרצינו — אבל אם המטרה הייתה לשמור עותק נקי לצד המקור לצורך השוואה, זו טעות שקטה. הפתרון: לבקש ‎`.copy()`‎ באופן מפורש.

In [2]:
raw_log2 = np.array([20.0, 18.8, -999.0, -999.0, -999.0, 5.2, 0.1])
cleaned = raw_log2[2:5].copy()   # עותק אמיתי
cleaned[:] = [17.6, 12.4, 8.9]

print("גולמי (לא השתנה):", raw_log2)
print("מנוקה:", cleaned)

גולמי (לא השתנה): [ 2.00e+01  1.88e+01 -9.99e+02 -9.99e+02 -9.99e+02  5.20e+00  1.00e-01]
מנוקה: [17.6 12.4  8.9]


```{image} ../../figures/7_5_view_copy.svg
:alt: A slice is a view of the same cells; .copy() makes new cells
:align: center
:class: only-light
```

```{image} ../../figures/7_5_view_copy-dark.svg
:alt: A slice is a view of the same cells; .copy() makes new cells
:align: center
:class: only-dark
```

### בדקו את עצמכם

```python
a = np.array([1, 2, 3])
b = a[1:]
b[0] = 99
print(a[1])
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מה תדפיס שורת ההדפסה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "99",
                "correct": true,
                "feedback": "נכון: פריסה של מערך NumPy מחזירה תצוגה, ולכן השינוי ב-b משנה גם את a."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא: b היא תצוגה של a, לא עותק."
            },
            {
                "answer": "1",
                "correct": false,
                "feedback": "לא: האיבר הראשון של b הוא האיבר השני של a, ואותו שינינו."
            },
            {
                "answer": "שגיאה",
                "correct": false,
                "feedback": "לא: זה קוד חוקי לחלוטין."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### תרגול עצמי: מצאו את השגיאה

הפעולה ‎`a.mean()`‎ מחזירה את הממוצע של איברי המערך ‎`a`‎; למשל, הממוצע של ארבע המהירויות בתא הבא הוא 14.35.

הקוד בתא הבא נועד ליצור גרסה "מנורמלת" של וקטור המהירויות (כל מהירות פחות הממוצע), בלי לפגוע במקור. הציפייה הייתה ששורה 4 תדפיס את המדידות המקוריות:

```
[12.  15.5  9.8 20.1]
```

אבל היא מדפיסה:

```
[-2.35  1.15 -4.55  5.75]
```

קראו את הקוד לפני שאתם מריצים אותו: באיזו שורה השגיאה? תקנו אותה בעזרת ‎`.copy()`‎.

In [ ]:
v_measured = np.array([12.0, 15.5, 9.8, 20.1])   # speeds (m/s)
v_normalized = v_measured[:]   # meant as an independent copy
v_normalized -= v_normalized.mean()
print(v_measured)

`````{admonition} פתרון
:class: dropdown, tip
השגיאה בשורה 2: ‎`v_measured[:]`‎ היא תצוגה של אותו מערך, לא עותק, ולכן החיסור בשורה 3 משנה גם את ‎`v_measured`‎. מתקנים את שורה 2:

```python
v_measured = np.array([12.0, 15.5, 9.8, 20.1])
v_normalized = v_measured.copy()
v_normalized -= v_normalized.mean()
print(v_measured)     # [12.  15.5  9.8 20.1]
print(v_normalized)   # [-2.35  1.15 -4.55  5.75]
```
`````

``````{admonition} תרגול נוסף
:class: dropdown, note
נתון מערך ‎`a`‎. שנו את שני האיברים הראשונים של תצוגה ‎`b = a[:2]`‎ ל-‎`[0, 0]`‎, ובדקו בעצמכם אם ‎`a`‎ השתנה:

```python
a = np.array([1, 2, 3, 4, 5])
b = a[:2]
# שנו את b כך שיהיה [0, 0], ואז הדפיסו את a
# b[:] = ...
# print(a)
```

`````{admonition} פתרון
:class: dropdown, tip
```python
b[:] = [0, 0]
print(a)   # [0 0 3 4 5]: a changed, because b is a view of a
```
`````
``````